In [ ]:
!pip install open-clip-torch

In [10]:
import json
import random
from pathlib import Path
from PIL import Image
from collections import defaultdict
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T

class SupervisedProductDataset(Dataset):
    def __init__(
        self,
        json_path: str,
        image_root: str = "",
        max_per_category: int = 1000,
        is_train: bool = True
    ):
        self.image_root = Path(image_root)
        self.is_train = is_train

        with open(json_path, "r") as f:
            raw_data = json.load(f)
        
        counts = defaultdict(int)
        self.data = []
        
        # --- 1. Platform Balancing Logic ---
        for item in raw_data:
            category = item.get("category")
            platform = item.get("platform")
        
            if not category or not platform:
                continue
            
            # Dynamic limits: Amazon gets 100%, Walmart gets 50%
            limit = max_per_category if platform == "Amazon" else (max_per_category * 0.75)
        
            key = (category, platform)
            if max_per_category == -1 or counts[key] < limit:
                self.data.append(item)
                counts[key] += 1

        # --- 2. Image Augmentation Pipelines ---
        # Base: Used for Anchor and Validation
        self.base_transform = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], 
                        std=[0.26862954, 0.26130258, 0.27577711])
        ])

        # Strong: Used for Positive Pair (Simulates varying real-world photos)
        self.aug_transform = T.Compose([
            T.RandomResizedCrop(224, scale=(0.8, 1.0)),
            T.RandomHorizontalFlip(p=0.5),
            # T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            T.ToTensor(),
            T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], 
                        std=[0.26862954, 0.26130258, 0.27577711])
        ])

    def __len__(self):
        return len(self.data)

    def _augment_title(self, title: str) -> str:
        """Keeps core brand/model intact, shuffles/drops descriptive trailing words."""
        if not title: return ""
        words = title.split()
        n = 4 # Protect the first 4 words
        
        if len(words) <= n:
            return title

        fixed = words[:n] # Corrected from your snippet (was words[n:])
        aug_tokens = words[n:]
        chunks = []
        i = 0

        # Group into semantic chunks
        while i < len(aug_tokens):
            chunk_size = random.choice([2, 3])
            chunks.append(" ".join(aug_tokens[i:i + chunk_size]))
            i += chunk_size

        # Shuffle and Drop
        if random.random() < 0.3:
            random.shuffle(chunks)
        chunks = [c for c in chunks if random.random() > 0.25]

        # Reassemble
        return " ".join(fixed + [w for chunk in chunks for w in chunk.split()])

    def __getitem__(self, idx: int):
        raw = self.data[idx]
        original_title = raw.get("title", "Unknown Product")

        # Load Image
        img_path = self.image_root / raw.get("image_path", "")
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            # Fallback for corrupted images to prevent training crashes
            image = Image.new('RGB', (224, 224), color='white')

        if self.is_train:
            return {
                "image_anchor": self.base_transform(image),
                "image_positive": self.aug_transform(image),
                "text_anchor": original_title,
                "text_positive": self._augment_title(original_title),
                "category": raw.get("category")
            }
        else:
            return {
                "image": self.base_transform(image),
                "text": original_title,
                "category": raw.get("category")
            }

# Data Splitting 

In [13]:
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import open_clip

# --- Configuration ---
MODEL_NAME = 'ViT-B-32'
PRETRAINED = 'laion2b_s34b_b79k'
JSON_PATH = "/kaggle/input/fyp-dataset/FINAL_Dataset_Phase1/final_data.json"
IMAGE_ROOT = "/kaggle/input/fyp-dataset"
BATCH_SIZE = 64  
MAX_PER_CAT = 2000

# 1. Initialize Dataset to get filtered data
full_dataset = SupervisedProductDataset(
    json_path=JSON_PATH,
    image_root=IMAGE_ROOT,
    max_per_category=MAX_PER_CAT,
    is_train=True
)

# 2. Extract categories for stratification
labels = [item['category'] for item in full_dataset.data]

# 3. Stratified Split (80% Train, 20% Val)
train_data, val_data = train_test_split(
    full_dataset.data, 
    test_size=0.25, 
    stratify=labels, 
    random_state=42
)

# 4. Create separate instances for Train (Augmented) and Val (Base)
train_dataset = SupervisedProductDataset(json_path=JSON_PATH, image_root=IMAGE_ROOT)
train_dataset.data = train_data
train_dataset.is_train = True

val_dataset = SupervisedProductDataset(json_path=JSON_PATH, image_root=IMAGE_ROOT)
val_dataset.data = val_data
val_dataset.is_train = False

# 5. DataLoaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE,          # Increase this until you get "Out of Memory"
    shuffle=True, 
    num_workers=4,           # Kaggle usually has 2-4 cores per GPU instance
    pin_memory=True,         # Faster CPU -> GPU transfer
    persistent_workers=True, # Keeps workers alive between epochs
    drop_last=True           # Prevents slow-down on the final small batch
)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,  shuffle=False, num_workers=4)

print(f"Setup Complete: {len(train_dataset)} training items, {len(val_dataset)} validation items.")

Setup Complete: 19201 training items, 6401 validation items.


## Pre Evaluation

In [14]:
import torch.nn.functional as F
from tqdm import tqdm

def evaluate_retrieval(model, dataloader, device):
    model.eval()
    all_image_features = []
    all_text_features = []

    print("Extracting features for evaluation...")
    with torch.no_grad():
        for batch in tqdm(dataloader):
            # Use 'image' and 'text' keys from your val_loader
            images = batch["image"].to(device)
            texts = open_clip.tokenize(batch["text"]).to(device)

            # Encode
            image_feat = model.encode_image(images)
            text_feat = model.encode_text(texts)

            # Normalize
            image_feat /= image_feat.norm(dim=-1, keepdim=True)
            text_feat /= text_feat.norm(dim=-1, keepdim=True)

            all_image_features.append(image_feat)
            all_text_features.append(text_feat)

    # Concatenate all features
    image_features = torch.cat(all_image_features) # [N, 512]
    text_features = torch.cat(all_text_features)   # [N, 512]

    # Calculate Cosine Similarity Matrix (N x N)
    logits = text_features @ image_features.T
    
    # Ground truth: the diagonal (index i of text matches index i of image)
    ground_truth = torch.arange(len(text_features)).to(device)

    # Calculate Recall@K
    def recall_at_k(logits, ground_truth, k):
        _, top_k_indices = logits.topk(k, dim=1)
        # Check if ground_truth[i] is in top_k_indices[i]
        correct = top_k_indices.eq(ground_truth.view(-1, 1).expand_as(top_k_indices))
        return correct.any(dim=1).float().mean().item()

    r1 = recall_at_k(logits, ground_truth, 1)
    r5 = recall_at_k(logits, ground_truth, 5)

    return {"R@1": r1, "R@5": r5}

In [15]:
from PIL import Image
import warnings

# Suppress the console spam
warnings.filterwarnings("ignore", category=UserWarning, module="PIL.Image")

def safe_load_image(img_path):
    try:
        img = Image.open(img_path)
        # If the image has transparency (RGBA or Palette)
        if img.mode in ('RGBA', 'P') or (img.mode == 'P' and 'transparency' in img.info):
            img = img.convert('RGBA')
            # Create a white background
            background = Image.new('RGBA', img.size, (255, 255, 255))
            # Paste the image onto the white background
            img = Image.alpha_composite(background, img)
            
        return img.convert('RGB')
    except Exception:
        # Fallback for corrupted images
        return Image.new('RGB', (224, 224), color='white')

In [16]:
#  Load the PRETRAINED weights
device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, _ = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
model = model.to(device)

#  Perform Pre-Evaluation
baseline_metrics = evaluate_retrieval(model, val_loader, device)

print("\nZERO-SHOT BASELINE (Pre-Fine-Tuning):")
print(f"Recall@1: {baseline_metrics['R@1']*100:.2f}%")
print(f"Recall@5: {baseline_metrics['R@5']*100:.2f}%")

Extracting features for evaluation...


100%|██████████| 101/101 [00:32<00:00,  3.08it/s]


ZERO-SHOT BASELINE (Pre-Fine-Tuning):
Recall@1: 31.31%
Recall@5: 57.26%


In [17]:
import torch
import random
import open_clip

# --- 1. Load Pre-trained Model ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading pre-trained LAION model...")
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model = model.to(device)
model.eval()

# --- 2. Extract Features for the Validation Set ---
# We will use a subset (e.g., 500 items) to keep the test fast
sample_size = min(500, len(val_dataset.data))
test_pool = val_dataset.data[:sample_size]

print(f"Encoding {sample_size} validation products...")
pool_vectors = []
pool_titles = []

with torch.no_grad():
    for item in test_pool:
        # Load and preprocess
        img_path = val_dataset.image_root / item.get("image_path", "")
        img = safe_load_image(img_path) # Using our new safe loader
        img_tensor = preprocess(img).unsqueeze(0).to(device)
        
        txt_tensor = open_clip.tokenize([item.get("title", "")]).to(device)
        
        # Extract and Normalize
        img_feat = model.encode_image(img_tensor)
        img_feat /= img_feat.norm(dim=-1, keepdim=True)
        
        txt_feat = model.encode_text(txt_tensor)
        txt_feat /= txt_feat.norm(dim=-1, keepdim=True)
        
        # Fusion Vector (Matches your HNSW pipeline)
        fusion_vector = ((img_feat + txt_feat) / 2.0)
        fusion_vector /= fusion_vector.norm(dim=-1, keepdim=True)
        
        pool_vectors.append(fusion_vector)
        pool_titles.append(item.get("title", "Unknown"))

# Stack into a single matrix: Shape [N, 512]
database_matrix = torch.cat(pool_vectors)

# --- 3. Pick 2 Random Anchors and Find Top 5 ---
anchors = random.sample(range(sample_size), 2)

for anchor_idx in anchors:
    anchor_title = pool_titles[anchor_idx]
    anchor_vector = database_matrix[anchor_idx].unsqueeze(0) # Shape [1, 512]
    
    # Calculate Cosine Similarity against all items
    # Matrix multiplication of normalized vectors = Cosine Similarity
    similarities = (anchor_vector @ database_matrix.T).squeeze(0)
    
    # Get Top 6 (Top 1 is always the anchor itself, so we get 6 and skip the 1st)
    top_scores, top_indices = similarities.topk(6)
    
    print(f"\n🔍 ANCHOR PRODUCT: '{anchor_title[:70]}...'")
    print("-" * 80)
    
    # Start loop from 1 to skip the exact same product
    for i in range(1, 6):
        match_idx = top_indices[i].item()
        match_score = top_scores[i].item()
        match_title = pool_titles[match_idx]
        
        print(f"  {i}. [Score: {match_score*100:.1f}%] | {match_title[:70]}...")

Loading pre-trained LAION model...
Encoding 500 validation products...


KeyboardInterrupt: 

# Training Loop

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import open_clip

# --- 1. Hyperparameters ---
MODEL_NAME = 'ViT-B-32'
PRETRAINED = 'laion2b_s34b_b79k'
EPOCHS = 18
LR = 5e-5             # Slightly higher because 90% of the model is frozen
WEIGHT_DECAY = 0.2      
MAX_GRAD_NORM = 1.0     
EARLY_STOP_PATIENCE = 2 

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- 2. Load Model & FREEZE BASE LAYERS ---
print("Loading model and freezing base layers...")
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
model = model.to(device)

# Lock everything
model.requires_grad_(False) 

# Unfreeze only the vital projection heads and temperature
model.logit_scale.requires_grad_(True)
if hasattr(model, 'visual') and hasattr(model.visual, 'proj') and model.visual.proj is not None:
    model.visual.proj.requires_grad_(True)
if hasattr(model, 'text_projection') and model.text_projection is not None:
    model.text_projection.requires_grad_(True)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Training only {trainable_params:,} parameters to prevent catastrophic forgetting.")

# --- 3. Loss & Optimizer Setup ---
# This native loss automatically handles the N x N matrix of positives and negatives
loss_fn = open_clip.ClipLoss()

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = GradScaler()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)

# --- 4. The Training Loop ---
print(f"\nStarting Fine-Tuning on {device}...")
best_r1 = 0.0
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    model.train()
    step_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    
    for batch in pbar:
        # We use the augmented positive views. Every other item in the batch acts as a negative!
        images = batch["image_positive"].to(device)
        texts = open_clip.tokenize(batch["text_positive"]).to(device)

        with autocast(device_type='cuda'):
            # Forward pass: Returns image embeddings, text embeddings, and the learnable temperature
            image_features, text_features, logit_scale = model(images, texts)
            
            # The InfoNCE matrix loss (Positives on diagonal, Negatives everywhere else)
            total_loss = loss_fn(image_features, text_features, logit_scale)

        optimizer.zero_grad()
        scaler.scale(total_loss).backward()
        
        # Gradient Clipping (Applied only to our unfrozen parameters)
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), MAX_GRAD_NORM)
        
        scaler.step(optimizer)
        scaler.update()

        step_loss += total_loss.item()
        pbar.set_postfix({"loss": f"{total_loss.item():.4f}", "lr": f"{scheduler.get_last_lr()[0]:.1e}"})

    scheduler.step()
    avg_train_loss = step_loss / len(train_loader)

    # --- 5. Validation Loop ---
    print(f"\nValidating Epoch {epoch+1}...")
    # Call your pre-existing evaluate_retrieval function
    val_metrics = evaluate_retrieval(model, val_loader, device)
    current_r1 = val_metrics["R@1"]
    current_r5 = val_metrics["R@5"]
    
    print(f" Loss: {avg_train_loss:.4f} |  Val R@1: {current_r1*100:.2f}% | Val R@5: {current_r5*100:.2f}%")

    # --- 6. Early Stopping ---
    if current_r1 > best_r1:
        best_r1 = current_r1
        epochs_without_improvement = 0
        print(f"New Best R@1! Saving model...")
        torch.save({
            'model_state_dict': model.state_dict(),
            'val_metrics': val_metrics,
            'epoch': epoch + 1
        }, "best_openclip_model.pt")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(f"Stopping early. Best R@1 was {best_r1*100:.2f}%")
            break

print("\nTraining finished. Your fine-tuned weights are saved safely.")

Loading model and freezing base layers...
Training only 655,361 parameters to prevent catastrophic forgetting.

Starting Fine-Tuning on cuda...


Epoch 1/18 [Train]: 100%|██████████| 300/300 [01:58<00:00,  2.54it/s, loss=0.2218, lr=5.0e-05]



Validating Epoch 1...
Extracting features for evaluation...


100%|██████████| 101/101 [00:33<00:00,  3.05it/s]


 Loss: 0.3322 |  Val R@1: 37.34% | Val R@5: 66.15%
New Best R@1! Saving model...


...

Epoch 7/18 [Train]: 100%|██████████| 300/300 [01:28<00:00,  3.38it/s, loss=0.1436, lr=3.8e-05]



Validating Epoch 7...
Extracting features for evaluation...


100%|██████████| 101/101 [00:33<00:00,  3.03it/s]


 Loss: 0.1442 |  Val R@1: 38.73% | Val R@5: 69.77%


Epoch 8/18 [Train]:  25%|██▌       | 75/300 [00:24<01:15,  2.97it/s, loss=0.1485, lr=3.4e-05]

## Post Evaluation

In [7]:
import torch
import open_clip

# --- 1. Setup Device and Base Model ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading base model architecture...")
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained=None)

# --- 2. Load Your Fine-Tuned Weights ---
print("Loading fine-tuned weights from 'best_openclip_model.pt'...")
checkpoint = torch.load("best_openclip_model.pt", map_location=device)

# Handle potential nested state_dicts (depending on how it was saved)
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(checkpoint)

model = model.to(device)
model.eval()

# --- 3. Run the Evaluation ---
print("\nRunning Final Evaluation on Validation Set...")
# Make sure your val_loader and evaluate_retrieval function from earlier are still in memory
final_metrics = evaluate_retrieval(model, val_loader, device)

# --- 4. Display Results ---
print("\n" + "="*40)
print("🏆 FINE-TUNED MODEL RESULTS")
print("="*40)
print(f"Final Recall@1: {final_metrics['R@1']*100:.2f}%")
print(f"Final Recall@5: {final_metrics['R@5']*100:.2f}%")
print("="*40)

Loading base model architecture...


Loading fine-tuned weights from 'best_openclip_model.pt'...

Running Final Evaluation on Validation Set...
Extracting features for evaluation...


100%|██████████| 49/49 [00:15<00:00,  3.13it/s]


🏆 FINE-TUNED MODEL RESULTS
Final Recall@1: 47.21%
Final Recall@5: 79.27%
